# Cooling-RAG Quickstart — End-to-End Reproduction

This notebook reproduces a single end-to-end run of the multi-granular RAG pipeline used in the paper:

> *Multi-Granular Retrieval-Augmented Generation for Evidence-Based Engineering Requirement Formulation in Ship Cooling Systems*

It is designed to work in **two modes**:

1. **Cached-replay mode** *(default — no API key needed, fully deterministic)*
   - Downloads the LLM response cache from Zenodo (a few MB)
   - All DeepSeek calls hit the cache and return the same answers we used to produce paper Table 3
2. **Live mode** *(requires `DEEPSEEK_API_KEY`)*
   - Calls the live DeepSeek API
   - Cache misses count against your API budget

Either mode produces a `shall`-statement requirement set, a 5-dimension quality gate report, and SysML-aligned XML, exportable as SysML 1.6-aligned XML (a custom importer is required for Cameo/Rhapsody; direct import is not provided in this release).

**Estimated wall-clock**: ≤ 3 minutes on a free Colab T4 instance in cached-replay mode.

## 1. Install dependencies

In [ ]:
!pip install -q sentence-transformers==5.5.1 faiss-cpu==1.13.2 rank-bm25==0.2.2 openai==2.37.0 python-docx pydantic langgraph>=0.2 python-dotenv lxml diskcache

## 2. Clone the code repository

Replace `<URL>` below with the GitHub URL provided in the paper's *Data Availability* section.

In [ ]:
!git clone https://github.com/<anonymized-during-review>/cooling-rag-multigranular.git
%cd cooling-rag-multigranular

## 3. Download KB, indexes, and LLM cache from Zenodo

All artefacts referenced by the paper are archived at the Zenodo DOI listed in the paper's *Data Availability* section. Replace `<ZENODO_RECORD>` below.

In [ ]:
!wget -q https://zenodo.org/record/<ZENODO_RECORD>/files/cooling_rag_data.tar.gz
!tar -xzf cooling_rag_data.tar.gz -C ./
# Should populate data/processed/, data/indexes/, data/llm_cache/

## 4. (Optional) Set your DeepSeek API key — live mode only

Skip this cell to run in cached-replay mode.

In [ ]:
import os
# os.environ['DEEPSEEK_API_KEY'] = 'sk-...'   # uncomment to enable live mode
print('Live mode' if os.environ.get('DEEPSEEK_API_KEY') else 'Cached-replay mode')

## 5. Build services and run one requirement

We use the same `PipelineServices` wiring as the paper's G4 configuration: multi-granular retrieval, profile scoring, group expansion, LLM evidence judge, engineering-requirement agent, and 5-dimension quality gate.

In [ ]:
from cooling_rag_mg import seed_everything
from cooling_rag_mg.pipeline import make_services, run_requirements_with_services
from cooling_rag_mg.engineering_requirement_agent import generate_engineering_requirements
from cooling_rag_mg.quality_gate import QualityGate

seed_everything(42)
services = make_services()

example_requirement = {
    'requirement_id': 'demo-001',
    'stakeholder': '船东代表',
    'requirement_text': '主机冷却水系统应具备足够的冗余配置，单台冷却水泵故障时仍能维持主机基本运行；冷却水温度应有过温报警，并能在异常工况下保护主机。'
}

payload = run_requirements_with_services([example_requirement], services=services, top_k=4)
agent_output = generate_engineering_requirements([example_requirement], payload)
req_set = agent_output['requirement_sets'][0]

qg = QualityGate()
for er in req_set['engineering_requirements']:
    er['quality_gate_report'] = qg.check(er).to_dict()

print('Generated', len(req_set['engineering_requirements']), 'engineering requirements.')

## 6. Inspect the output

In [ ]:
import json
for er in req_set['engineering_requirements']:
    print('━' * 60)
    print(f"[{er['id']}] {er['topic']}")
    print(f"  shall:    {er['shall_statement']}")
    print(f"  status:   {er['support_status']}")
    print(f"  evidence: {er.get('evidence_ids', [])}")
    print(f"  verify:   {er['verification_method']}")
    qg_report = er.get('quality_gate_report', {})
    print(f"  QG pass:  {qg_report.get('overall_passed')}  (score {qg_report.get('score')}/1.0)")

## 7. Export to SysML-aligned XML

In [ ]:
from cooling_rag_mg.sysml_exporter import export_requirement_set
from pathlib import Path

out_path = Path('demo-001.xml')
export_requirement_set(req_set, out_path)
print('SysML XML written to', out_path.resolve())
print('First 600 chars:')
print(out_path.read_text(encoding='utf-8')[:600])

## 8. Next steps

- To reproduce the full paper Table 3: `python scripts/26_run_three_rounds.py --groups G1 G2 G3 G4 A1 A2`
- To recompute metrics: `python scripts/22_compute_metrics.py`
- To run the per-requirement statistical analysis: `python scripts/27_statistics.py`
- To run external baselines (long-context LLM, vanilla RAG, KB scaling): `python scripts/28_baselines.py all`
- To launch the interactive Streamlit UI: `streamlit run app.py`

Every numerical value in the paper is keyed to a `manifest.json` in its corresponding batch directory under `data/search_results/`, so every reported number is traceable to a specific run with model version, seed, and full hyperparameter set recorded.